# Workbook 2 · Which requirements size the aircraft? Speed, hot day, and the engines

A requirement only costs weight if it is **binding**. This workbook shows how to tell, and what happens when you change a requirement that is not binding (nothing)
versus one that is (a lot). Then it changes the engines: the one input the Halo treats as fixed.

**Run time:** about 2-3 minutes (four solves).

In [1]:
import sys
from pathlib import Path
repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(repo_root / "tutorials"))
from sizing_workbench import *
from dataclasses import replace
import re
base = baseline_variant()
print("binding in the baseline:")
for label in sorted({re.sub(r' \d+/\d+:', ':', x) for x in base.result.binding}):
    print("   ", label)

binding in the baseline:
    bus-out hover: motor torque_Nm
    cn_beta_per_rad
    engine-out hover: battery end voltage_V
    engine-out hover: generator torque_Nm
    engine-out hover: generator_gearbox power_input_W
    engine-out hover: turboshaft power_shaft_W
    hot-day hover: heat_exchanger power_heat_W
    hot-day hover: turboshaft power_shaft_W
    hover: gearbox power_input_W
    hover: propulsor power_shaft_W
    rotor_radius_m
    static_margin


## 1. Predict first

Before solving: is the **210 kt maximum-speed** requirement in that list? (Look for `max_speed:`.) If it is not, lowering it to 190 kt should change nothing.

In [2]:
speed_is_binding = any(label.startswith("max_speed") for label in base.result.binding)
print("max speed binding in the baseline:", speed_is_binding)
slower = size("max speed 190 kt", requirements=replace(base.requirements, velocity_max_m_s=190 * u.knot))
compare(base, slower)
check("relaxing a non-binding requirement leaves the design unchanged",
      (not speed_is_binding) and abs(slower.result.mass_takeoff_kg - base.result.mass_takeoff_kg) < 1.0)

max speed binding in the baseline: False


[max speed 190 kt: sized in 38 s from 'baseline': 15,134 lb]
                                            baseline        max speed 190 kt
take-off weight [lb]                          15,134          15,134 (-0.0%)
empty weight [lb]                             11,087          11,087 (-0.0%)
payload [lb]                                   1,984           1,984 (+0.0%)
fuel [lb]                                      2,063           2,063 (+0.0%)
battery [kWh]                                   69.5            69.5 (-0.0%)
wing area [m2]                                 23.24           23.24 (-0.0%)
span [m]                                       11.93           11.93 (-0.0%)
rotor radius [m]                               4.825           4.825 (-0.0%)
disk loading [kg/m2]                            46.9            46.9 (+0.0%)
cruise speed [kt]                                161             161 (-0.0%)
cruise L/D                                      9.05            9.05 (-0.0%)
motor rating pe

Nothing changed. On this design the 210 kt dash has slack; the engines are sized out (they are fixed at 1,120 hp) and the drive is set by hovers. **A requirement
you can relax for free is a requirement that is not binding.** Asking "what does 190 kt buy us?" is answered by the binding list before any solve.

## 2. A requirement that does bind: the hot-day hover

The hot-day hover (4,000 ft, 95 °F, at the destination) is in the binding list (it sizes the heat exchanger and loads the turboshafts). Drop it:

In [3]:
no_hot = size("no hot-day hover", requirements=replace(base.requirements, hover_hot_day=False))
compare(base, no_hot)
weight_changes(base, no_hot)
binding_changes(base, no_hot)
check("dropping the hot-day hover saves weight", no_hot.result.mass_takeoff_kg < base.result.mass_takeoff_kg - 50)

[no hot-day hover: sized in 32 s from 'baseline': 14,447 lb]
                                            baseline        no hot-day hover
take-off weight [lb]                          15,134          14,447 (-4.5%)
empty weight [lb]                             11,087          10,458 (-5.7%)
payload [lb]                                   1,984           1,984 (+0.0%)
fuel [lb]                                      2,063           2,005 (-2.8%)
battery [kWh]                                   69.5           62.4 (-10.3%)
wing area [m2]                                 23.24           22.39 (-3.7%)
span [m]                                       11.93           11.71 (-1.8%)
rotor radius [m]                               4.825           4.715 (-2.3%)
disk loading [kg/m2]                            46.9            46.9 (-0.0%)
cruise speed [kt]                                161             161 (+0.0%)
cruise L/D                                      9.05            8.98 (-0.8%)
motor rating pe

That one requirement is worth hundreds of pounds. This is the kind of number to bring to a requirements discussion: *the 95 °F destination hover costs X lb; is it
worth it?*

## 3. The engines

The Halo's engines are **bought, not designed**: 2 × 1,120 hp, fixed (`power_rated_turboshaft_fixed_W`). Two variants:

- a bigger off-the-shelf engine, 1,300 hp;
- a **rubber** engine (`None`): the optimizer sizes the engine power too, with mass and efficiency from the AeroSandbox regressions (tutorial 06).

In [4]:
bigger = size("engines 1,300 hp", assumptions=replace(base.assumptions, power_rated_turboshaft_fixed_W=1300 * u.hp))
rubber = size("rubber engines", assumptions=replace(base.assumptions, power_rated_turboshaft_fixed_W=None))
compare(base, bigger, rubber)
print(f"\nrubber engine power: {rubber.result.design.power_rated_turboshaft_W / u.hp:,.0f} hp each")

[engines 1,300 hp: sized in 49 s from 'baseline': 15,263 lb]


[rubber engines: sized in 49 s from 'baseline': 14,531 lb]
                                            baseline        engines 1,300 hp          rubber engines
take-off weight [lb]                          15,134          15,263 (+0.9%)          14,531 (-4.0%)
empty weight [lb]                             11,087          11,149 (+0.6%)          10,578 (-4.6%)
payload [lb]                                   1,984           1,984 (+0.0%)           1,984 (+0.0%)
fuel [lb]                                      2,063           2,130 (+3.3%)           1,968 (-4.6%)
battery [kWh]                                   69.5           56.4 (-18.9%)           77.9 (+12.1%)
wing area [m2]                                 23.24           23.40 (+0.7%)           22.50 (-3.2%)
span [m]                                       11.93           11.97 (+0.3%)           11.73 (-1.6%)
rotor radius [m]                               4.825           4.846 (+0.4%)           4.729 (-2.0%)
disk loading [kg/m2]            

Read it carefully. The bigger engine makes the aircraft **heavier**: extra power is only useful where the engines bind, and the extra engine mass is paid
everywhere. The rubber engine finds the power that minimizes take-off weight, smaller than 1,120 hp here, and the aircraft gets lighter. That gap measures the
**cost of buying a fixed engine** that is larger than this mission needs.

In [5]:
check("a bigger fixed engine is not free", bigger.result.mass_takeoff_kg > base.result.mass_takeoff_kg)
check("a rubber engine is at least as light as the fixed one", rubber.result.mass_takeoff_kg <= base.result.mass_takeoff_kg + 1.0)
binding_changes(base, rubber)

PASS  a bigger fixed engine is not free
PASS  a rubber engine is at least as light as the fixed one
now binding (new):
    climb: generator temperature_C
    max_speed: turboshaft power_shaft_W
    take-off hover: generator torque_Nm
    take-off hover: generator_gearbox power_input_W
    take-off hover: turboshaft power_shaft_W
no longer binding:
    -
still binding:
    bus-out hover: motor torque_Nm
    cn_beta_per_rad
    engine-out hover: battery end voltage_V
    engine-out hover: generator torque_Nm
    engine-out hover: generator_gearbox power_input_W
    engine-out hover: turboshaft power_shaft_W
    hot-day hover: heat_exchanger power_heat_W
    hot-day hover: turboshaft power_shaft_W
    hover: gearbox power_input_W
    hover: propulsor power_shaft_W
    rotor_radius_m
    static_margin


## 4. Your turn

Pick a requirement from the baseline's binding list and relax it a little, or pick one that is **not** in the list and predict the answer before you run.
Some fields of `HaloRequirements` to try: `altitude_hover_m`, `thrust_to_weight_hover`, `duration_engine_out_hover_s`, `altitude_ceiling_m`,
`temperature_hover_hot_K`, `duration_loiter_s`.

In [6]:
my_requirements = replace(base.requirements, duration_engine_out_hover_s=45.0)    # <- edit: 60 s in the baseline
mine = size("my requirement change", requirements=my_requirements)
compare(base, mine)
binding_changes(base, mine)

[my requirement change: the warm start from 'baseline' failed; trying the full starting-point strategy (can take minutes)]


[my requirement change: sized in 236 s from 'multistart': 14,558 lb]
                                            baseline   my requirement change
take-off weight [lb]                          15,134          14,558 (-3.8%)
empty weight [lb]                             11,087          10,551 (-4.8%)
payload [lb]                                   1,984           1,984 (+0.0%)
fuel [lb]                                      2,063           2,023 (-1.9%)
battery [kWh]                                   69.5           61.2 (-12.0%)
wing area [m2]                                 23.24           22.53 (-3.1%)
span [m]                                       11.93           11.74 (-1.5%)
rotor radius [m]                               4.825           4.733 (-1.9%)
disk loading [kg/m2]                            46.9            46.9 (-0.0%)
cruise speed [kt]                                161             160 (-0.5%)
cruise L/D                                      9.05            8.97 (-0.9%)
motor r

<details><summary>What to expect for a 45 s engine-out hover</summary>

The engine-out hover sizes the battery by voltage at its end (tutorial 07), so a shorter hover ends at a higher SOC and higher voltage: the pack shrinks, and
everything that carried it shrinks too. The battery line in `compare` should drop noticeably. Try `altitude_ceiling_m` next: if the ceiling is not binding, the
design will not move at all.
</details>

## What you practised

- Read the binding list before solving: non-binding requirements are free to change; binding ones cost weight.
- Requirement trades are one `replace` on `HaloRequirements`; engine choices are assumptions (`power_rated_turboshaft_fixed_W`, `None` for a rubber engine).
- Bigger is not better unless the extra capability is what binds.

Next: **Workbook 3**, technology and architecture.

In [7]:
check_summary()

4 of 4 checks passed
